In [39]:
from pathlib import Path
import requests
import base64
from io import BytesIO
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import base64
from io import BytesIO
from PIL import Image, ImageEnhance
from IPython.display import Image as DisplayImage
from azure.identity import (
    AuthenticationRecord,
    DeviceCodeCredential,
    TokenCachePersistenceOptions,
)
from dotenv import load_dotenv
import os
load_dotenv()

TENANT_ID = os.getenv("AZURE_TENANT_ID")
CLIENT_ID = os.getenv("LOCAL_GRAPH_CLIENT_ID")
TOP = 50

In [80]:
# The delegated permission requested from Microsoft Graph.
GRAPH_SCOPE = "https://graph.microsoft.com/Mail.Read"

# Stores non-secret information identifying your signed-in account.
AUTH_RECORD_PATH = Path.home() / ".graph-local-auth-record"

# Microsoft Graph endpoint for your own inbox.
GRAPH_URL = (
    "https://graph.microsoft.com/v1.0"
    "/me/mailFolders/sentitems/messages"
    "?$search=\"to:timesheets@mobileappscompany.com hasAttachments:true\""
)

def create_credential() -> DeviceCodeCredential:
    """
    Create a credential that signs in with device-code authentication.

    The first execution asks you to sign in. Later executions normally
    reuse the encrypted local token cache.
    """

    authentication_record = None

    if AUTH_RECORD_PATH.exists():
        authentication_record = AuthenticationRecord.deserialize(
            AUTH_RECORD_PATH.read_text(encoding="utf-8")
        )

    cache_options = TokenCachePersistenceOptions(
        name="graph-local-inbox-cache"
    )

    credential = DeviceCodeCredential(
        tenant_id=TENANT_ID,
        client_id=CLIENT_ID,
        authentication_record=authentication_record,
        cache_persistence_options=cache_options,
    )

    # On the first run, authenticate and remember which account was used.
    if authentication_record is None:
        authentication_record = credential.authenticate(
            scopes=[GRAPH_SCOPE]
        )

        AUTH_RECORD_PATH.write_text(
            authentication_record.serialize(),
            encoding="utf-8",
        )

        # Allow only your local macOS user to read the file.
        AUTH_RECORD_PATH.chmod(0o600)

    return credential

In [81]:
def read_inbox() -> None:
    credential = create_credential()

    # Obtain a Microsoft Graph access token.
    token = credential.get_token(GRAPH_SCOPE)

    response = requests.get(
        GRAPH_URL,
        headers={
            "Authorization": f"Bearer {token.token}",
            "Accept": "application/json",
        },
        params={
            "$top": TOP
        },
        timeout=30,
    )

    if not response.ok:
        print("Microsoft Graph returned an error:")
        print(response.status_code)
        print(response.text)
        response.raise_for_status()

    messages = response.json().get("value", [])

    print(f"\nSuccessfully retrieved {len(messages)} messages.\n")

    return messages

In [82]:
messages = read_inbox()


Successfully retrieved 41 messages.



In [86]:
messages[0]

{'@odata.etag': 'W/"CQAAABYAAACRzdadMySeT5oa5m5xYMtpAAOhj51K"',
 'id': 'AAMkADE5OWMzZmQ5LTdhOTAtNDU2YS1iZjk2LTkxMjU3MzJlNTJlMABGAAAAAADd_MWgW_HITKHZyP_fstmdBwCRzdadMySeT5oa5m5xYMtpAAAAAAEJAACRzdadMySeT5oa5m5xYMtpAAEuASWvAAA=',
 'createdDateTime': '2024-02-14T00:31:16Z',
 'lastModifiedDateTime': '2026-09-22T01:32:22Z',
 'changeKey': 'CQAAABYAAACRzdadMySeT5oa5m5xYMtpAAOhj51K',
 'categories': [],
 'receivedDateTime': '2024-02-14T00:32:08Z',
 'sentDateTime': '2024-02-14T00:32:05Z',
 'hasAttachments': True,
 'internetMessageId': '<DM6PR07MB626790A66A3491478179DD20B34E2@DM6PR07MB6267.namprd07.prod.outlook.com>',
 'subject': 'Approved Timesheet',
 'bodyPreview': '',
 'importance': 'normal',
 'parentFolderId': 'AQMkADE5OQBjM2ZkOS03YTkwLTQ1NmEtYmY5Ni05MTI1NzMyZTUyZTAALgAAA934xaBb4chModnI-5_y2Z0BAJHN1p0zJJ5PmhrmbnFgy2kAAAIBCQAAAA==',
 'conversationId': 'AAQkADE5OWMzZmQ5LTdhOTAtNDU2YS1iZjk2LTkxMjU3MzJlNTJlMAAQAK30WR_8cWdJjdKDWhOC9eQ=',
 'conversationIndex': 'AQHaXt0frfRZH7xxZ0mN0oNaE4L15A==',
 'i

In [99]:
credential = create_credential()

# Obtain a Microsoft Graph access token.
token = credential.get_token(GRAPH_SCOPE)

attachments_file = {}

for message in messages:
    attachments = requests.get(
        f"https://graph.microsoft.com/v1.0/me/messages/{message['id']}/attachments",
        headers={"Authorization": f"Bearer {token.token}"}
    )
    attachments_json = attachments.json()
    for attachment in attachments_json["value"]:
        if message['id'] not in attachments_file:
            attachments_file[message['id']] = []
        attachments_file[message['id']].append(attachment)

# write attachments_file to a file
with open("attachments.json", "w") as f:
    json.dump(attachments_file, f, indent=4)



In [84]:
# write the retrieved messages to a file
with open("messages.json", "w", encoding="utf-8") as f:
    import json
    json.dump(messages, f, ensure_ascii=False, indent=4)

In [16]:
import base64
import os
from io import BytesIO
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()   

# Enable LangSmith tracing when its API key is configured.
if os.getenv("LANGSMITH_API_KEY"):
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_PROJECT", "ai-for-timesheets")


model = init_chat_model("gpt-5-mini")

# --------------------------------------------------
# 1. Pydantic schema
# --------------------------------------------------

class ConsultantDetails(BaseModel):
    consultant_name: str = Field(
        description="Full name of the consultant"
    )

    date_hours_per_day: dict = Field(
        description="Number of working hours per day for as many dates as available in the timesheet. If a day has no hours, include it with a value of 0",
        examples=[{"2024-06-01": "<hours_worked_per_day>", "2024-06-02": "<hours_worked_per_day>", "2024-06-03": "<hours_worked_per_day>"}]
    )

    vendor_name: str = Field(
        description="Consultant's vendor name - tends to be a small lesser known consulting company"
    )
    end_client_name: str = Field(
        description="Name of the end client - tends to be a large well known company"
    )
    timesheet_pay_period_start_date: str = Field(
        description="The start date of the period covered by the timesheet, typically in the format 'YYYY-MM-DD'"
    )
    timesheet_pay_period_end_date: str = Field(
        description="The end date of the period covered by the timesheet, typically in the format 'YYYY-MM-DD'"
    )

    timesheet_status: str = Field(
        description="Status of the consultant's timesheet",
        examples=["Submitted", "Approved", "Unknown"]
    )

 
System_Prompt=""" Act as a timesheet reviewer. You should carefully review the timesheet image and extract
 the consultant and timesheet. 

 Be extremely thorough and precise when extracting the information. Especially with tabular data with rows and columns. 
 Carefully check each row and column to ensure accuracy at extraction.

  Identify Orientation: Determine whether the schedule uses days as Columns and hours as Rows, or vice versa. 
  Match & Associate: Map each specific day directly to its associated hour or activity block across the identified grid structure.

    Extract all available information from this image.

                Identify:
                - Consultant name
                - Date hours per day
                - Vendor name
                - End client name
                - Timesheet pay period start date
                - Timesheet pay period end date
                - Timesheet status

                Return the information using the required
                structured Pydantic schema.

                
DATE SOURCE PRIORITY:
- Extract timesheet dates and hours only from the embedded A.C.COY timesheet form.
- Ignore the Outlook title, email Subject, Sent date, Expires date, and message body when determining timesheet dates.
- The email metadata may conflict with the form. The form is authoritative.
- Inspect the Week Ending field and every row in the Date column.
- If the form month is unreadable, return it as ambiguous instead of inferring it from email metadata.

ROW-LOCKED EXTRACTION:

Treat the area between each pair of horizontal grid lines as one independent
row. Read the Date, Hourly Regular, and Totals cells from that SAME horizontal
band before moving to the next row.

A number belongs only to the date in its own row. Never move, pack, resequence,
or distribute hours to form a conventional workweek. Blank rows may occur at
the beginning, middle, or end.

Preserve a blank cell as null. Mark an unreadable cell as illegible rather than
guessing. Verify each nonzero value against the duplicated value in that row's
Totals column.


Do not guess missing information or edit extracted data (even if it doesn't look correct) only report what is actually present nothing more nothing less.
                """

# --------------------------------------------------
# 3. Create structured-output agent
# --------------------------------------------------

agent = create_agent(
    model=model,
    system_prompt=System_Prompt,
    response_format=ConsultantDetails
)


# --------------------------------------------------
# 4. Convert image to Base64
# --------------------------------------------------

def image_to_base64(image_object) -> str:
    output_buffer = BytesIO()
    image_object.save(output_buffer, format='PNG')
    return base64.b64encode(output_buffer.getvalue()).decode("utf-8")


# --------------------------------------------------
# 5. Send Base64 image to agent
# --------------------------------------------------

def extract_consultant(image_base64: str):

    message = {
        "role": "user",
        "content": [
            
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{image_base64}",
                    "detail": "high"
                }
            }
        ]
    }

    result = agent.invoke(
        {
            "messages": [message]
        }
    )

    return result["structured_response"]



In [17]:
def read_extract_timesheet_hours(image_path):

    image = Image.open(image_path)

    # increase image quality
    image = image.convert("RGBA")

    # Enhance sharpness
    sharpness_factor = 1.5  # Increase sharpness by 50%
    enhancer_sharpness = ImageEnhance.Sharpness(image)
    sharpened_image = enhancer_sharpness.enhance(sharpness_factor)

    # Enhance contrast
    contrast_factor = 1.5  # Increase contrast by 50%
    enhancer_contrast = ImageEnhance.Contrast(sharpened_image)
    enhanced_image = enhancer_contrast.enhance(contrast_factor)

    # Save the enhanced image to a BytesIO object to display it
    output_buffer = BytesIO()
    # convert any image to PNG format
    enhanced_image.save(output_buffer, format="PNG")
    output_buffer.seek(0)

    # Display the enhanced image
    DisplayImage(output_buffer.getvalue())


    timesheet_details = extract_consultant(
        image_to_base64(enhanced_image)
    )
    
    print("Consultant Name:", timesheet_details.consultant_name)
    print("Date Hours/Day:", timesheet_details.date_hours_per_day)
    print("Vendor Name:", timesheet_details.vendor_name)
    print("End Client Name:", timesheet_details.end_client_name)
    print("Timesheet Pay Period Start Date:", timesheet_details.timesheet_pay_period_start_date)
    print("Timesheet Pay Period End Date:", timesheet_details.timesheet_pay_period_end_date)
    print("Timesheet Status:", timesheet_details.timesheet_status)

    return timesheet_details

In [18]:
read_extract_timesheet_hours("timesheet1-sakshi.png")

Consultant Name: Sakshi Shetty
Date Hours/Day: {'2026-09-08': 8.0, '2026-09-09': 8.0, '2026-09-10': 8.0, '2026-09-11': 8.0}
Vendor Name: ELEVATE
End Client Name: illegible
Timesheet Pay Period Start Date: 2026-09-08
Timesheet Pay Period End Date: 2026-09-13
Timesheet Status: Approved


ConsultantDetails(consultant_name='Sakshi Shetty', date_hours_per_day={'2026-09-08': 8.0, '2026-09-09': 8.0, '2026-09-10': 8.0, '2026-09-11': 8.0}, vendor_name='ELEVATE', end_client_name='illegible', timesheet_pay_period_start_date='2026-09-08', timesheet_pay_period_end_date='2026-09-13', timesheet_status='Approved')

In [19]:
read_extract_timesheet_hours("timesheet2-sakshi.png")

Consultant Name: Sakshi Shetty
Date Hours/Day: {'2026-08-24': 8.0, '2026-08-25': 8.0, '2026-08-26': 8.0, '2026-08-27': 8.0, '2026-08-28': 8.0}
Vendor Name: illegible
End Client Name: illegible
Timesheet Pay Period Start Date: 2026-08-24
Timesheet Pay Period End Date: 2026-08-28
Timesheet Status: Approved


ConsultantDetails(consultant_name='Sakshi Shetty', date_hours_per_day={'2026-08-24': 8.0, '2026-08-25': 8.0, '2026-08-26': 8.0, '2026-08-27': 8.0, '2026-08-28': 8.0}, vendor_name='illegible', end_client_name='illegible', timesheet_pay_period_start_date='2026-08-24', timesheet_pay_period_end_date='2026-08-28', timesheet_status='Approved')

In [20]:
read_extract_timesheet_hours("blurry_image.jpeg")

Consultant Name: Clive Dias
Date Hours/Day: {'2026-09-23': 0, '2026-09-24': 8, '2026-09-25': 8, '2026-09-26': 8, '2026-09-27': 8, '2026-09-28': 8, '2026-09-29': 0}
Vendor Name: A.C.COY
End Client Name: Westinghouse
Timesheet Pay Period Start Date: 2026-09-23
Timesheet Pay Period End Date: 2026-09-29
Timesheet Status: Approved


ConsultantDetails(consultant_name='Clive Dias', date_hours_per_day={'2026-09-23': 0, '2026-09-24': 8, '2026-09-25': 8, '2026-09-26': 8, '2026-09-27': 8, '2026-09-28': 8, '2026-09-29': 0}, vendor_name='A.C.COY', end_client_name='Westinghouse', timesheet_pay_period_start_date='2026-09-23', timesheet_pay_period_end_date='2026-09-29', timesheet_status='Approved')

In [21]:
read_extract_timesheet_hours("timesheet_clean_example.png")

Consultant Name: Dennis Huynh
Date Hours/Day: {'2026-09-06': 0.0, '2026-09-07': 0.0, '2026-09-08': 8.0, '2026-09-09': 8.0, '2026-09-10': 8.0, '2026-09-11': 8.0, '2026-09-12': 0.0}
Vendor Name: Tech Wish, LLC(UALY)
End Client Name: Navy Federal Credit Union
Timesheet Pay Period Start Date: 2026-09-06
Timesheet Pay Period End Date: 2026-09-12
Timesheet Status: Approved


ConsultantDetails(consultant_name='Dennis Huynh', date_hours_per_day={'2026-09-06': 0.0, '2026-09-07': 0.0, '2026-09-08': 8.0, '2026-09-09': 8.0, '2026-09-10': 8.0, '2026-09-11': 8.0, '2026-09-12': 0.0}, vendor_name='Tech Wish, LLC(UALY)', end_client_name='Navy Federal Credit Union', timesheet_pay_period_start_date='2026-09-06', timesheet_pay_period_end_date='2026-09-12', timesheet_status='Approved')